# Fund Liquidity Stress and LMT Workflow

This notebook assesses the selected fund’s ability to meet redemption pressure at the valuation date and illustrates how selected liquidity management tools may respond under a dynamic redemption path.

The workflow separates three layers. First, it applies deterministic redemption stress scenarios sourced from the fund’s risk management policy. These scenarios represent point-in-time liability shocks, including investor concentration events such as the largest holder or three largest holders redeeming. Second, it compares each redemption shock with the asset-side liquidity profile of the fund, based on cash, liquid holdings and liquidation buckets. Third, it applies selected LMT mechanics, including gates, anti-dilution adjustments, notice-period effects and suspension indicators, where redemption pressure exceeds defined policy thresholds.

The dynamic redemption path is presented as an optional simulation layer. It is not the primary liquidity stress scenario. Its purpose is to show how redemption pressure could evolve across future dealing dates after LMT mechanics are applied, including deferred redemptions and backlog effects.

References for building this notebook can be found [here](../../docs/notebooks_notes/liquidity_references.md)

In [ ]:
# connect to SQL database
from src.data.database   import get_engine
ENGINE  = get_engine()

# fund to be analysed
FUND = "UCITS_Balanced"

# In this repo all computation refer to the same date
from src.config import VALUATION_DATE
VALUATION_DATE

## 1. Fund Liquidity Terms and Policy Inputs

This section loads the fund-level inputs used in the liquidity workflow: fund profile, redemption terms, liquidity policy settings, and investor-base data. These inputs define the fund’s liquidity obligations and monitoring setup before the portfolio and stress scenario are analysed.

In [ ]:
# Display fund overview combined with liquidity monitoring parameters
import src.ui.liquidity_calibration_display as lcd
lcd.display_fund_liquidity_overview(
    fund_id=FUND,
    engine=ENGINE,
    export_id="01",
)

## 2. Portfolio Liquidity Profile

This section estimates how quickly the selected fund’s holdings could be converted into cash under the project’s liquidity classification. This creates the asset-side liquidity view used later in the stress test.


Days-to-liquidate estimates position-level liquidation timelines.
Liquidity buckets classify assets by expected settlement speed.

In [ ]:
# Liquidity profile — plot only
from src.ui.liquidity_plot import plot_liquidity_profile
from src.data.enrichment import get_risk_ready_df
from src.risk.risk_utils import compute_liquidity_profile

FUND = 'UCITS_Balanced'

# Query and enrich positions
risk_df = get_risk_ready_df(ENGINE, FUND, VALUATION_DATE)
NAV = risk_df['market_value_eur'].sum()

# Compute liquidity profile
liquidity_pct_adv = 0.25
liq = compute_liquidity_profile(risk_df, pct_adv=liquidity_pct_adv)
bucket_full = liq['bucket_full']
risk_df_liq = liq['risk_df_liq']

# Display liquidity profile chart only
plot_liquidity_profile(bucket_full, FUND, metric='pct_nav_abs', valuation_date=VALUATION_DATE, export_id="02")


## 3. Investor Base and Redemption Scenario Inputs

This section loads the liability-side inputs used in the redemption stress workflow.

The investor register supports investor concentration checks. The RMP redemption scenarios define the point-in-time redemption shocks applied to the valuation-date NAV before LMT mechanics are considered.


In [ ]:
# Load investor registers and calibration inputs
from src.data.reference_data import load_investor_and_calibration_data

# Load calibration data for UCITS_Balanced
data = load_investor_and_calibration_data(FUND)
investor_inputs = data['investor_inputs']
calibration_inputs = data['calibration_inputs']
calibration_config = data['calibration_config']

# Display top 5 investors
lcd.display_top_investors(investor_inputs, FUND, VALUATION_DATE, top_n=5)


### Investor Redemption Model
 we use beta motel for redemptiosn on a 

In [ ]:

# Compute redemption scenarios
from src.computation.liquidity_calibration import (
    compute_redemption_scenarios,
)
scenarios_result = compute_redemption_scenarios(investor_inputs, calibration_config)
scenarios_result

## 4. Point-in-Time Redemption Stress

This section applies one selected redemption scenario to the fund’s valuation-date portfolio. It compares the redemption amount with the liquid assets available under the project liquidity buckets. The purpose is to test whether the fund could meet a defined redemption shock before applying any LMT mechanics.

In [ ]:
# Redemption stress — use existing function
import src.ui.print_html_utils as phtml
from src.data.reference_data import get_stress_testing_params
from src.pipeline.liquidity_policy import load_redemption_scenarios_from_rmp

# Load redemption scenarios from RMP
REDEMPTION_SCENARIOS = load_redemption_scenarios_from_rmp(FUND)

# Get stress testing params
params = get_stress_testing_params(FUND, calibration_inputs, redemption_scenario="Large")
notice_period_days = params['notice_days']

# Display using existing function
phtml.display_redemption_stress(
    FUND,
    notice_period_days,
    REDEMPTION_SCENARIOS,
    NAV,
    risk_df_liq,
    valuation_date=VALUATION_DATE,
    export_id="03",
)


## 5. Redemption Path and Pressure Before LMTs

This section applies the RMP redemption scenarios to the valuation-date NAV before any liquidity management tools are considered.

The purpose is to show the raw redemption pressure first: the redemption amount under each scenario, the size of the shock relative to NAV, and the liquidity available before applying gates, deferrals, swing pricing, anti-dilution adjustments or suspension indicators.

This provides the baseline for the later LMT section. LMTs should be interpreted as a response to the stress result, not as part of the initial redemption shock.

### Investor Redemption Path Model

This section builds a dynamic redemption path for the LMT mechanics analysis. It is separate from the point-in-time redemption stress scenarios.

The point-in-time scenarios test immediate liquidity coverage at the valuation date. The redemption path answers a different question: how redemption pressure may evolve across future dealing dates when gates, deferred redemptions, swing / anti-dilution adjustments or suspension indicators are assessed.

For normal months, each investor type receives a beta-distributed redemption draw centred on its normal monthly redemption assumption. The beta distribution is used because redemption rates are bounded between 0 and 1.

For stress months, the model uses the stressed redemption assumption directly, so the stress event is not weakened by random sampling.

The aggregate fund-level redemption rate is calculated as the investor-weighted sum of the monthly redemption rates across investor types.



In [ ]:
import pandas as pd

# Available liquidity before LMTs
# Adjust bucket names if your liquidity profile uses different labels.
available_liquidity_eur = risk_df_liq.loc[
    risk_df_liq["liquidity_bucket"].isin(["Cash", "0-5 days", "5-30 days"]),
    "market_value_eur",
].sum()

pre_lmt_rows = []

for redemption_rate, scenario_name in REDEMPTION_SCENARIOS:
    redemption_amount_eur = NAV * redemption_rate
    shortfall_eur = max(redemption_amount_eur - available_liquidity_eur, 0)

    pre_lmt_rows.append({
        "scenario": scenario_name,
        "nav_eur": NAV,
        "redemption_rate": redemption_rate,
        "redemption_amount_eur": redemption_amount_eur,
        "redemption_pct_nav": redemption_amount_eur / NAV,
        "available_liquidity_eur": available_liquidity_eur,
        "liquidity_coverage_ratio": available_liquidity_eur / redemption_amount_eur
        if redemption_amount_eur else None,
        "shortfall_before_lmt_eur": shortfall_eur,
        "shortfall_pct_nav": shortfall_eur / NAV,
    })

pre_lmt_redemption_coverage = pd.DataFrame(pre_lmt_rows)

pre_lmt_redemption_coverage


## 6. Liquidity Management Tools Analysis

Now we apply the  LMT mechanics layer to the dynamic path: fund’s gate, swing pricing or anti-dilution, and suspension settings to the redemption scenario. The trigger matrix shows when each tool becomes active. In a redemption scenario, swing pricing or anti-dilution is treated as a cost adjustment to redeeming investors, protecting remaining investors from liquidity costs linked to large outflows.

In [ ]:
# Display LMT parameters from calibration
lcd.display_lmt_parameters(calibration_inputs, FUND, export_id="04")

In [ ]:
# LMT Trigger Analysis
from src.pipeline.lmt_trigger_analysis import (
    build_lmt_analysis_inputs,
    run_lmt_trigger_analysis,
    prepare_scenarios_data,
)
# Build LMT inputs from calibration
lmt_inputs = build_lmt_analysis_inputs(
    FUND,
    calibration_inputs,
    calibration_config,
)

# Run LMT trigger analysis
analysis = run_lmt_trigger_analysis(
    ENGINE,
    FUND,
    lmt_inputs,
    VALUATION_DATE,
)

df_analysis = analysis['df_analysis']

# Prepare scenarios for display
scenarios_data = prepare_scenarios_data(scenarios_result)

# Display redemption scenarios and LMT analysis
lcd.display_lmt_summary(df_analysis, FUND, VALUATION_DATE, export_id="05")

# Display 3 separate LMT plots
lcd.plot_lmt_paid_deferred_backlog(df_analysis, FUND, VALUATION_DATE, export_id="06")
lcd.plot_lmt_nav_evolution(df_analysis, FUND, VALUATION_DATE, export_id="07")
lcd.plot_lmt_flags(df_analysis, FUND, VALUATION_DATE, export_id="08")